### Блок 8. Forecasting: лаги, сезонность и подготовка признаков

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

import matplotlib.dates as mdates
from matplotlib.transforms import blended_transform_factory
from statsmodels.tsa.stattools import acf, pacf

#### Подготовка и вспомогательный функции 

In [ ]:
fc_data = pd.read_excel('data/final_data.xlsx')
fc_data["rep_date"] = pd.to_datetime(fc_data["rep_date"])
fc_data = fc_data.sort_values(["country", "hs", "rep_date"]).reset_index(drop=True)

shock_date = pd.Timestamp("2022-02-01")

save_dir = "figures/eda_block_8"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
def human_format(x, pos):
    abs_x = abs(x)
    if abs_x >= 1e9:
        return f"{x/1e9:.1f}B"
    elif abs_x >= 1e6:
        return f"{x/1e6:.1f}M"
    elif abs_x >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"

def apply_style(ax, title, ylabel=None, xlabel=None, shade_post=False, top=False, post_sanctions=True):
    ax.set_title(title, fontsize=16, fontweight="bold", pad=20)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=12, labelpad=15)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=12, labelpad=15)

    ax.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_alpha(0.5)
    ax.spines["bottom"].set_alpha(0.5)
    ax.tick_params(axis="both", labelsize=11)

    if shade_post:
        # правая граница именно текущей оси
        x_left = mdates.date2num(shock_date)
        x_right = ax.get_xlim()[1]

        # красим до конца графика
        ax.axvspan(x_left, x_right, alpha=0.08, color="#B22222")
        ax.axvline(shock_date, color="#8B0000", linestyle="--", linewidth=1.4, alpha=0.8)

        # X в координатах данных, Y в долях оси
        trans = blended_transform_factory(ax.transData, ax.transAxes)

        # подпись чуть левее правого края красной зоны
        x_text = x_right - 0.02 * (x_right - x_left)

        if top:
            if post_sanctions:
                ax.text(
                    x_text, 0.94,
                    "Постсанкционный период",
                    transform=trans,
                    color="#8B0000",
                    fontsize=10,
                    ha="right",
                    va="top",
                    linespacing=1.0
                )
        else: 
            if post_sanctions: 
                ax.text(
                    x_right - 0.1 * (x_right - x_left), 0.04,
                    "Постсанкционный период",
                    transform=trans,
                    color="#8B0000",
                    fontsize=10,
                    ha="right",
                    va="bottom",
                    linespacing=1.0
                )

def save_figure(fig, filename):
    fig.savefig(os.path.join(save_dir, f"{filename}.png"), dpi=400, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def weighted_mean(x, value_col, weight_col):
    temp = x[[value_col, weight_col]].dropna()
    if temp.empty or temp[weight_col].sum() == 0:
        return np.nan
    return np.average(temp[value_col], weights=temp[weight_col])


def rank_binned_summary(data, factor_col, target_col="country_import_value", n_bins=8, cap_q=0.99):
    temp = data.dropna(subset=[factor_col, target_col]).copy()

    cap_target = temp[target_col].quantile(cap_q)
    temp["target_plot"] = temp[target_col].clip(upper=cap_target)

    temp["factor_rank"] = temp[factor_col].rank(method="first")
    temp["factor_bin"] = pd.qcut(
        temp["factor_rank"],
        q=n_bins,
        labels=[f"Q{i}" for i in range(1, n_bins + 1)]
    )

    summary = (
        temp.groupby("factor_bin", observed=False)
        .agg(
            median_target=("target_plot", "median"),
            q25_target=("target_plot", lambda x: x.quantile(0.25)),
            q75_target=("target_plot", lambda x: x.quantile(0.75)),
            mean_factor=(factor_col, "mean"),
            obs=("target_plot", "size")
        )
        .reset_index()
    )
    return summary


def plot_binned_relationship(summary, title, xlabel, filename, line_color="#2F5D8A", band_color="#C7D3DD"):
    x = np.arange(len(summary))

    fig, ax = plt.subplots(figsize=(11.8, 7))

    ax.bar(
        x,
        summary["q75_target"] - summary["q25_target"],
        bottom=summary["q25_target"],
        width=0.62,
        color=band_color,
        alpha=0.60,
        edgecolor="none",
        label="25–75 перцентиль"
    )

    ax.plot(
        x,
        summary["median_target"],
        color=line_color,
        linewidth=2.6,
        marker="o",
        markersize=6.5,
        label="Медиана"
    )

    for xi, val in zip(x, summary["median_target"]):
        ax.text(
            xi,
            val + summary["median_target"].max() * 0.03,
            human_format(val, None),
            ha="center",
            va="bottom",
            fontsize=9,
            color=line_color
        )

    ymin = summary["q25_target"].min()
    ymax = summary["q75_target"].max()
    yrange = ymax - ymin
    ax.set_ylim(max(0, ymin - 0.12 * yrange), ymax + 0.18 * yrange)

    ax.set_xticks(x)
    ax.set_xticklabels(summary["factor_bin"], fontsize=11)
    ax.yaxis.set_major_formatter(FuncFormatter(human_format))

    apply_style(
        ax,
        title=title,
        xlabel=xlabel,
        ylabel="Импорт страны, долл. США"
    )

    ax.legend(frameon=False, fontsize=11, loc="upper left")
    save_figure(fig, filename)